# Registration publication figures: saved dataframes only

Use this notebook to edit and export figures without rerunning the experiment.

First run the **Export plotting data** cell in the experimental notebook once. It writes a portable folder containing compressed CSV dataframes and a JSON manifest. After that, this notebook needs only that folder and the plotting helpers: no raw datasets, model checkpoints, autoencoders, CAST, PASTE, PASTE2, or PyTorch.

The bundle contains full-precision scores, timings, case metadata, original measurement coordinates with CCF division labels, and registered coordinates for every saved method. Plot edits do not modify the saved tables.


In [1]:
from pathlib import Path
import json
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.text import Text
import sys

ROOT = Path.cwd().resolve()
while not ((ROOT/'pyproject.toml').is_file() and (ROOT/'paper').is_dir()):
    if ROOT==ROOT.parent:
        raise RuntimeError('Repository root not found')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from paper.workflows import registration_plot_data, standardized_publication
pd.options.display.float_format = '{:.4g}'.format

# Set an explicit exported folder to pin a particular result set.
BUNDLE_PATH = None
bundle_rows = []
for path in (ROOT/'notebooks/checkpoints/registration_publication_controls').glob('*/reports/plot_data/*/manifest.json'):
    saved = json.loads(path.read_text())
    bundle_rows.append(dict(bundle=path.parent.name, complete=saved['complete'],
                            path=str(path.parent), source_report=saved['source_report']))
bundles = pd.DataFrame(bundle_rows, columns=['bundle','complete','path','source_report'])
display(bundles)
if BUNDLE_PATH is None:
    if bundles.empty:
        raise FileNotFoundError('Run the Export plotting data cell in the experiment notebook once; then return here.')
    completed = bundles[bundles.complete]
    candidates = completed if not completed.empty else bundles
    BUNDLE_PATH = candidates.sort_values('bundle').iloc[-1].path
BUNDLE = Path(BUNDLE_PATH).expanduser().resolve()
manifest, tables = registration_plot_data.load_plot_bundle(BUNDLE)
print('Selected bundle:', BUNDLE)
print('Complete experiment:', manifest['complete'])
FIGURES = ROOT/'figures/registration_publication_plotting'/BUNDLE.name


,bundle,complete,path,source_report


FileNotFoundError: Run the Export plotting data cell in the experiment notebook once; then return here.

## Dataframes

These are ordinary pandas dataframes, so you can inspect, summarize, or copy them before changing the plots. Spatial tables use a stable measurement index within each case and side. Registered coordinates refer to query measurements in that same order.

Do not confuse missing historical timings with zero runtime. The runtime table reports measured fresh fits only where those times were recorded.


In [ ]:
scores = tables['scores']
availability = tables['availability']
case_table = tables['cases']
measurements = tables['measurements']
registered = tables['registered']
timings = tables['timings']
for name, frame in tables.items():
    print(name, frame.shape, list(frame.columns))
display(scores.head())
display(timings.head())


## Main publication figure

Edit the settings below and rerun this cell only. It reloads the plotting helper so edits to its layout code are picked up without restarting the kernel.

The figure retains natural Allen registration, controlled recovery, and BARseq unequal coverage. Quantitative evaluation panels retain all saved methods. The optional expression columns can be enabled independently of model fitting.


In [ ]:
importlib.reload(standardized_publication)
EXAMPLE_METHOD = 'Splat rigid'
MAX_MEASUREMENT_LINKS = 60
INCLUDE_COSINE = False
FIGURE_SIZE = (11.5, 9)
FONT_FAMILY = 'Arial'
FONT_SCALE = 1.0
SAVE_FIGURE = True
# Optional per-axis title overrides; see the axis inventory displayed below.
TITLE_OVERRIDES = {}

if not manifest['complete']:
    display(availability.loc[~availability.available])
    raise RuntimeError('This is an incomplete result bundle. Complete the missing experiment fits before the main comparison.')
plot_cases, plot_maps = registration_plot_data.plotting_inputs(dict(
    tables, cases=case_table, measurements=measurements, registered=registered))
figure = standardized_publication.publication_figure(
    plot_cases, plot_maps, scores, methods=manifest['methods'],
    example_method=EXAMPLE_METHOD, include_cosine=INCLUDE_COSINE,
    measurement_links=True, maximum_links=MAX_MEASUREMENT_LINKS)
figure.set_size_inches(*FIGURE_SIZE)
for artist in figure.findobj(match=Text):
    artist.set_fontfamily(FONT_FAMILY)
    artist.set_fontsize(artist.get_fontsize()*FONT_SCALE)
for index, title in TITLE_OVERRIDES.items():
    figure.axes[index].set_title(title, fontfamily=FONT_FAMILY)
display(pd.DataFrame([dict(axis=i, title=ax.get_title()) for i,ax in enumerate(figure.axes)]))
if SAVE_FIGURE:
    FIGURES.mkdir(parents=True, exist_ok=True)
    stem = 'registration_expression' if INCLUDE_COSINE else 'registration_main'
    for extension in ('pdf','png'):
        figure.savefig(FIGURES/f'{stem}.{extension}', dpi=300, bbox_inches='tight')
plt.show()
plt.close(figure)


## Runtime figure

Edit this cell independently. Timings describe different workloads, not hardware-matched algorithmic speedups. Splat/CAST family totals include shared fitting and both affine variants; rigid pose times exclude field fitting. PASTE2 includes preprocessing and overlap selection. MLP fields and rigid pose are separate stages.

Evaluation cases are plotted separately by dataset. Saved machine configurations are displayed below. An absent timing is not plotted as zero.


In [ ]:
importlib.reload(registration_plot_data)
RUNTIME_ROLE = 'evaluation'
RUNTIME_FIGURE_SIZE = (12, 4.6)
print('Saved hardware configurations:')
for hardware in manifest['hardware']:
    print(json.loads(hardware))
if len(manifest['hardware'])>1:
    print('Multiple hardware configurations: stratify before making speed claims.')
display(timings.groupby(['dataset','task','role','stage']).measured_fresh_minutes.agg(
    ['count','median','min','max']))
selected = timings[timings.role==RUNTIME_ROLE]
if selected.measured_fresh_minutes.notna().any():
    runtime = registration_plot_data.runtime_figure(timings, role=RUNTIME_ROLE)
    runtime.set_size_inches(*RUNTIME_FIGURE_SIZE)
    if SAVE_FIGURE:
        FIGURES.mkdir(parents=True, exist_ok=True)
        for extension in ('pdf','png'):
            runtime.savefig(FIGURES/f'registration_runtime.{extension}', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(runtime)
else:
    print('No measured fresh runtimes for this role.')


## Portable results

Keep the complete exported bundle folder together. It can be moved or shared: the manifest's original report path is provenance only and is never used to load models or data.

To edit the figure implementation, change `paper/workflows/standardized_publication.py` or the runtime plot in `paper/workflows/registration_plot_data.py`, then rerun the corresponding plotting cell. No fitting or scoring is performed by this notebook.
